# Cosine stability and prediction consistency
Reorganized only; no experiment cells executed during creation. Existing results were copied with byte-hash verification.

In [ ]:
def compute_feature_similarity(paired, clean_features, clean_predictions, test_indices, y_test):
    # CELL 08 — Cosine stability and prediction agreement
    feature_rows, prediction_rows = [], []
    for name, conditions in paired.items():
        for condition, data in conditions.items():
            positions = data['positions']
            similarity = F.cosine_similarity(clean_features[name][positions].float(), data['features'].float(), dim=1)
            frame = pd.DataFrame({
                'backbone':name, 'condition':condition, 'pair_index':np.arange(len(positions)),
                'official_test_index':test_indices[positions.numpy()],
                'content_label':y_test[positions].numpy(), 'cosine_similarity':similarity.numpy()})
            feature_rows.append(frame)
            for model_name, logits in data['logits'].items():
                table = frame.copy()
                before = clean_predictions[model_name][positions]
                after = logits.argmax(1)
                table['model'] = model_name
                table['clean_prediction'] = before.numpy()
                table['transformed_prediction'] = after.numpy()
                table['prediction_unchanged'] = (before == after).numpy()
                prediction_rows.append(table)
    stability_pairs = pd.concat(feature_rows, ignore_index=True)
    prediction_pairs = pd.concat(prediction_rows, ignore_index=True)
    stability_summary = stability_pairs.groupby(['backbone','condition'], as_index=False).agg(
        mean_cosine=('cosine_similarity','mean'), std_cosine=('cosine_similarity','std'), n_pairs=('pair_index','size'))
    prediction_summary = prediction_pairs.groupby(['model','condition'], as_index=False).agg(
        mean_cosine=('cosine_similarity','mean'), prediction_consistency=('prediction_unchanged','mean'))
    conditional_stability = prediction_pairs.groupby(['model','condition','prediction_unchanged'], as_index=False).agg(
        mean_cosine=('cosine_similarity','mean'), n_pairs=('pair_index','size'))
    # Each translation direction has the same 500 images: this is an equal-direction mean.
    translation_stability = stability_pairs[stability_pairs['condition'].str.startswith('translation_')].copy()
    translation_stability['displacement'] = translation_stability['condition'].str.split('_').str[1].astype(int)
    translation_stability = translation_stability.groupby(['backbone','displacement'], as_index=False).agg(
        mean_cosine=('cosine_similarity','mean'), n_image_direction_pairs=('pair_index','size'))
    display(stability_summary.round(4))
    display(prediction_summary.round(4))
    display(conditional_stability.round(4))

    return stability_pairs, prediction_pairs, stability_summary, prediction_summary, conditional_stability, translation_stability

